# Ontario Housing Price Prediction

This is the coursework notebook reconstructed from the original HTML export. Saved outputs were cleared because the source preview contains property-level records. The original model includes a synthetic estimate derived from the target as an input, which leaks the answer into the model. See the project README for details before interpreting any scores.

# Ontario Housing Price Prediction Project

This notebook uses the Ontario housing dataset and builds regression models to predict housing prices.
It evaluates performance against:

Listing price as the benchmark proxy for the Zolo-style estimate
Sold price as the actual target

Dataset priority

data_on.csv
cleaned_ontario_sold.csv
submission_files/code/data/cleaned_ontario_sold.csv

Import Libraries

In this section, we import all necessary libraries for data analysis, visualization, and machine learning.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

Load Dataset

The Ontario housing dataset is loaded into a pandas DataFrame for analysis.

In [ ]:
# Load Ontario dataset
possible_files = [
    "data_on.csv",
    "cleaned_ontario_sold.csv",
    "submission_files/code/data/cleaned_ontario_sold.csv",
]

file_path = None
for f in possible_files:
    if os.path.exists(f):
        file_path = f
        break

if file_path is None:
    raise FileNotFoundError("Could not find Ontario dataset file.")

df = pd.read_csv(file_path)
df.columns = df.columns.str.strip().str.lower()

print("Loaded file:", file_path)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Data Preview

We inspect the dataset to understand its structure and key features.

Data Understanding

This step checks column names and identifies missing values in the dataset.

In [ ]:
# Inspect missing values
print("Missing values by column:")
print(df.isnull().sum().sort_values(ascending=False))

Data Preparation

The dataset is cleaned by converting important columns to numeric format and handling missing values.

In [ ]:
# -----------------------------
# CLEAN COLUMN NAMES
# -----------------------------
df.columns = df.columns.str.strip().str.lower()

# -----------------------------
# CONVERT KEY COLUMNS TO NUMERIC
# -----------------------------
def clean_num(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce"
    )

key_cols = ["price", "property-beds", "property-baths", "property-sqft", "latitude", "longitude"]
for col in key_cols:
    if col in df.columns:
        df[col] = clean_num(df[col])

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
if "property-beds" in df.columns and "property-baths" in df.columns:
    df["total_rooms"] = df["property-beds"] + df["property-baths"]
    df["bath_per_bed"] = df["property-baths"] / (df["property-beds"] + 1)

if "property-sqft" in df.columns and "property-beds" in df.columns:
    df["sqft_per_bed"] = df["property-sqft"] / (df["property-beds"] + 1)

# -----------------------------
# TARGET + BENCHMARK
# -----------------------------
target_col = "price"

# simulate Zolo-style benchmark since this file does not include listing_price
df["zolo_estimate"] = df["price"] * np.random.uniform(0.95, 1.05, size=len(df))
benchmark_col = "zolo_estimate"

# -----------------------------
# KEEP MODEL COLUMNS
# -----------------------------
model_cols = [
    "price",
    "zolo_estimate",
    "property-beds",
    "property-baths",
    "property-sqft",
    "latitude",
    "longitude",
    "total_rooms",
    "bath_per_bed",
    "sqft_per_bed"
]

existing_model_cols = [col for col in model_cols if col in df.columns]
numeric_df = df[existing_model_cols].copy()

# remove missing rows
numeric_df = numeric_df.dropna()

print("Columns used for modeling:")
print(numeric_df.columns.tolist())
print("Shape after cleaning:", numeric_df.shape)

# -----------------------------
# FEATURES / TARGET
# -----------------------------
feature_cols = [col for col in numeric_df.columns if col not in [target_col]]
X = numeric_df[feature_cols].copy()
y = numeric_df[target_col].copy()

print("Number of features:", len(feature_cols))
print(feature_cols)

In [ ]:
# Correlation with sold price
corrs = numeric_df.corr(numeric_only=True)[target_col].sort_values(ascending=False)
print(corrs)

In [ ]:
# Features and target
feature_cols = [col for col in numeric_df.columns if col != target_col]
X = numeric_df[feature_cols].copy()
y = numeric_df[target_col].copy()

print("Number of features:", len(feature_cols))
print(feature_cols)
print("X shape:", X.shape)
print("y shape:", y.shape)

Train-Test Split

The dataset is split into training and testing sets to evaluate model performance.

In [ ]:
# -----------------------------
# TRAIN TEST SPLIT
# -----------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

zolo_test = numeric_df.loc[y_test.index, "zolo_estimate"]

# scale for linear/ridge
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

def threshold_accuracy(preds, actual, threshold):
    return np.mean(np.abs(preds - actual) / actual <= threshold) * 100

for name, model in models.items():
    if name in ["Linear Regression", "Ridge Regression"]:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    acc_zolo = threshold_accuracy(preds, zolo_test, 0.10)
    acc_actual = threshold_accuracy(preds, y_test, 0.15)

    print("\n-----------------------------")
    print(name)
    print(f"RMSE: {rmse:,.2f}")
    print(f"MAE: {mae:,.2f}")
    print(f"R²: {r2:.4f}")
    print(f"Within ±10% of Zolo benchmark: {acc_zolo:.2f}%")
    print(f"Within ±15% of actual price: {acc_actual:.2f}%")

Model Training
Multiple regression models are trained, including Linear Regression, Ridge Regression, and Random Forest.

In [ ]:
def percent_within_threshold(preds, actual, threshold):
    preds = np.array(preds)
    actual = np.array(actual)
    safe_actual = np.where(actual == 0, np.nan, actual)
    pct_error = np.abs(preds - actual) / safe_actual
    return np.nanmean(pct_error <= threshold) * 100

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

benchmark_test = numeric_df.loc[y_test.index, "zolo_estimate"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42)
}

results = []

for name, model in models.items():
    if name in ["Linear Regression", "Ridge Regression"]:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    within_benchmark_10 = percent_within_threshold(preds, benchmark_test, 0.10)
    within_actual_15 = percent_within_threshold(preds, y_test, 0.15)

    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Within_10pct_of_Benchmark": within_benchmark_10,
        "Within_15pct_of_Actual": within_actual_15
    })

results_df = pd.DataFrame(results).sort_values(by="R2", ascending=False)
results_df

In [ ]:
# Save results if you want them for the report
results_df.to_csv("model_results.csv", index=False)
print("Saved model_results.csv")

In [ ]:
# Actual vs predicted plot for the best model by R2
best_model_name = results_df.iloc[0]["Model"]
print("Best model:", best_model_name)

if best_model_name == "Linear Regression":
    best_model = LinearRegression().fit(X_train_scaled, y_train)
    best_preds = best_model.predict(X_test_scaled)
elif best_model_name == "Ridge Regression":
    best_model = Ridge(alpha=1.0).fit(X_train_scaled, y_train)
    best_preds = best_model.predict(X_test_scaled)
else:
    best_model = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_train, y_train)
    best_preds = best_model.predict(X_test)

plt.figure(figsize=(7, 5))
plt.scatter(y_test, best_preds, alpha=0.6)
plt.xlabel("Actual Sold Price")
plt.ylabel("Predicted Price")
plt.title(f"Actual vs Predicted Prices ({best_model_name})")
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution
residuals = y_test - best_preds

plt.figure(figsize=(7, 5))
plt.hist(residuals, bins=30)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title(f"Residual Distribution ({best_model_name})")
plt.tight_layout()
plt.show()

Correlation Analysis

A heatmap is used to visualize relationships between features and housing price.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr_cols = ["price", "property-beds", "property-baths", "property-sqft", 
             "latitude", "longitude", "total_rooms", "bath_per_bed", "sqft_per_bed"]

existing_corr_cols = [col for col in corr_cols if col in numeric_df.columns]

corr_matrix = numeric_df[existing_corr_cols].corr()

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Key Housing Features")
plt.tight_layout()
plt.show()

In [ ]:
results_df

Model Visualization

The relationship between actual and predicted values is visualized using a scatter plot.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

# Train best model again
best_model = Ridge(alpha=1.0)
best_model.fit(X_train_scaled, y_train)
best_preds = best_model.predict(X_test_scaled)

# Scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_preds, alpha=0.6)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Housing Prices")

# ideal line
min_val = min(y_test.min(), best_preds.min())
max_val = max(y_test.max(), best_preds.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.tight_layout()
plt.show()